In [18]:
import torch
import pandas as pd
from esm.pretrained import esm1b_t33_650M_UR50S  # ← one of the available loader funcs

# 1) Load model & alphabet
model, alphabet = esm1b_t33_650M_UR50S()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

# 2) Read your CSV
in_csv = "/Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/Data_processing/Targeton_allseq_meta.csv"
df = pd.read_csv(in_csv)

# 3) Prepare sequences
seqs = [(str(i), seq) for i, seq in enumerate(df["target_region"].fillna(""))]

# 4) Tokenize
batch_converter = alphabet.get_batch_converter()
labels, strs, tokens = batch_converter(seqs)
tokens = tokens.to(device)

# 5) Forward pass
with torch.no_grad():
    # For esm1b_t33_650M_UR50S the final representation layer index is 33
    results = model(tokens, repr_layers=[33])

# 6) Extract & pool
embs = results["representations"][33].mean(dim=1).cpu().numpy()

# 7) Attach & expand
df["single_embedding"] = list(embs)
emb_cols = [f"emb_{i}" for i in range(embs.shape[1])]
df[emb_cols] = pd.DataFrame(embs, index=df.index)

# 8) Save result
out_csv = "/Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/Data_processing/Targeton_allseq_meta_with_esm_emb.csv"
df.to_csv(out_csv, index=False)
print(f"Saved embeddings (dim={embs.shape[1]}) to {out_csv}")



FileNotFoundError: [Errno 2] No such file or directory: 'Targeton_allseq_meta.csv'

In [16]:
import esm.pretrained
import inspect

# List all public callables in esm.pretrained
model_loaders = [
    name for name, obj in inspect.getmembers(esm.pretrained, inspect.isfunction)
    if not name.startswith("_")
]
print("Available ESM model loader functions:\n", "\n".join(model_loaders))


Available ESM model loader functions:
 esm1_t12_85M_UR50S
esm1_t34_670M_UR100
esm1_t34_670M_UR50D
esm1_t34_670M_UR50S
esm1_t6_43M_UR50S
esm1b_t33_650M_UR50S
esm1v_t33_650M_UR90S
esm1v_t33_650M_UR90S_1
esm1v_t33_650M_UR90S_2
esm1v_t33_650M_UR90S_3
esm1v_t33_650M_UR90S_4
esm1v_t33_650M_UR90S_5
esm2_t12_35M_UR50D
esm2_t30_150M_UR50D
esm2_t33_650M_UR50D
esm2_t36_3B_UR50D
esm2_t48_15B_UR50D
esm2_t6_8M_UR50D
esm_if1_gvp4_t16_142M_UR50
esm_msa1_t12_100M_UR50S
esm_msa1b_t12_100M_UR50S
esmfold_v0
esmfold_v1
has_emb_layer_norm_before
load_hub_workaround
load_model_and_alphabet
load_model_and_alphabet_core
load_model_and_alphabet_hub
load_model_and_alphabet_local
load_regression_hub


In [24]:
import pandas as pd

df = pd.read_csv(in_csv)
print(df.columns.tolist())
print(df.head(2))


['Targeton', 'primer_type', 'primer_bases', 'product_size_bp', 'primer_score', 'tm_c', 'computed_tm_c', 'computed_gc_content', 'gRNA', 'off_target_summary_data', 'target_region']
  Targeton      primer_type              primer_bases  product_size_bp  \
0     ROCX  ROCX_1_LibAmp_R  TGTACCAGCAATGTTGGCTTTATC              NaN   
1     CLZX  CLZX_1_LibAmp_F        CCAAGCTTCCCGAGGGGT              NaN   

   primer_score  tm_c  computed_tm_c  computed_gc_content  \
0           NaN   NaN            NaN                  NaN   
1           NaN   NaN            NaN                  NaN   

                       gRNA            off_target_summary_data  \
0  CACCGCTTCATTCAGCACTGGCTC  {0: 1, 1: 1, 2: 3, 3: 19, 4: 194}   
1  CACCGCTGACGGACTTCATTCAAG    {0: 1, 1: 0, 2: 0, 3: 3, 4: 53}   

                                       target_region  
0  aggcaccatttaagaatcacacacaaacaagtgggacacatagtcc...  
1  ttgtagcgccaggaaggggacaggtgctgagactaggcctgcctct...  


In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from esm.pretrained import esm1b_t33_650M_UR50S
from esm.data import BatchConverter

# ─── CONFIG ─────────────────────────────────────────────────────
in_csv      = "/Users/ds39/Documents/Sunny/MAVE/RD_projects/Benchling_SelectedTables/Data_processing/Targeton_allseq_meta.csv"
output_dir  = "./embeddings_chunks"
final_csv   = "./all_embeddings.csv"

chunk_size  = 500    # number of rows per chunk
batch_size  = 32     # ESM forward‐pass batch
max_len     = 1024   # truncate raw seq to 1024 (BatchConverter will cap at 1022 + specials)
seq_columns = ["gRNA", "primer_bases", "target_region"]

os.makedirs(output_dir, exist_ok=True)

# ─── LOAD MODEL ─────────────────────────────────────────────────
model, alphabet = esm1b_t33_650M_UR50S()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

batch_converter = BatchConverter(
    alphabet,
    truncation_seq_length = 1022  # so final tokens ≤1024
)

# ─── EMBEDDING FUNCTION ─────────────────────────────────────────
def embed_columns(df_chunk):
    N = len(df_chunk)
    for col in seq_columns:
        seqs = df_chunk[col].fillna("").astype(str).str.upper().tolist()
        all_embs = []
        print(f" Embedding {col!r} for {N} rows…", flush=True)

        for start in range(0, N, batch_size):
            end   = min(start + batch_size, N)
            batch = [(str(i), seq[:max_len])
                     for i, seq in enumerate(seqs[start:end], start)]
            _, _, tokens = batch_converter(batch)
            tokens = tokens.to(device)

            with torch.no_grad():
                out = model(tokens, repr_layers=[33])

            embs = out["representations"][33].mean(dim=1).cpu().numpy()
            all_embs.append(embs)
            print(f"  • {col}: rows {start}–{end} done", flush=True)

        all_embs = np.vstack(all_embs)  # shape [N, D]
        df_chunk[f"{col}_emb"] = list(all_embs)
        D = all_embs.shape[1]
        emb_cols = [f"{col}_emb_{i}" for i in range(D)]
        df_chunk[emb_cols] = pd.DataFrame(all_embs, columns=emb_cols, index=df_chunk.index)

# ─── MAIN: CHUNK, EMBED, SAVE ────────────────────────────────────
df_all = pd.read_csv(in_csv)
total  = len(df_all)

for i in range(0, total, chunk_size):
    j        = min(i + chunk_size, total)
    chunk_df = df_all.iloc[i:j].copy()
    print(f"\nProcessing chunk rows {i}–{j}…", flush=True)
    embed_columns(chunk_df)
    out_path = os.path.join(output_dir, f"chunk_{i}_{j}.csv")
    chunk_df.to_csv(out_path, index=False)
    print(f" ✔ Saved chunk to {out_path}", flush=True)

# ─── STITCH ALL CHUNKS TOGETHER ──────────────────────────────────
print("\nCombining all chunks…", flush=True)
chunk_files = sorted(os.listdir(output_dir))
combined = pd.concat(
    [pd.read_csv(os.path.join(output_dir, fn)) for fn in chunk_files],
    ignore_index=True
)
combined.to_csv(final_csv, index=False)
print("✅ All done — final file:", final_csv)




Processing chunk rows 0–500…
 Embedding 'gRNA' for 500 rows…
  • gRNA: rows 0–32 done
  • gRNA: rows 32–64 done
  • gRNA: rows 64–96 done
  • gRNA: rows 96–128 done
  • gRNA: rows 128–160 done
  • gRNA: rows 160–192 done
  • gRNA: rows 192–224 done
  • gRNA: rows 224–256 done
  • gRNA: rows 256–288 done
  • gRNA: rows 288–320 done
  • gRNA: rows 320–352 done
  • gRNA: rows 352–384 done
  • gRNA: rows 384–416 done
  • gRNA: rows 416–448 done
  • gRNA: rows 448–480 done
  • gRNA: rows 480–500 done


/var/folders/88/r4q5_g595xs0mq3119k4nxy40000gq/T/ipykernel_51640/4019768598.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_chunk[emb_cols] = pd.DataFrame(all_embs, columns=emb_cols, index=df_chunk.index)
/var/folders/88/r4q5_g595xs0mq3119k4nxy40000gq/T/ipykernel_51640/4019768598.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_chunk[emb_cols] = pd.DataFrame(all_embs, columns=emb_cols, index=df_chunk.index)
/var/folders/88/r4q5_g595xs0mq3119k4nxy40000gq/T/ipykernel_51640/4019768598.py:56: PerformanceWarning: DataF

 Embedding 'primer_bases' for 500 rows…


/var/folders/88/r4q5_g595xs0mq3119k4nxy40000gq/T/ipykernel_51640/4019768598.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_chunk[emb_cols] = pd.DataFrame(all_embs, columns=emb_cols, index=df_chunk.index)
/var/folders/88/r4q5_g595xs0mq3119k4nxy40000gq/T/ipykernel_51640/4019768598.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_chunk[emb_cols] = pd.DataFrame(all_embs, columns=emb_cols, index=df_chunk.index)
/var/folders/88/r4q5_g595xs0mq3119k4nxy40000gq/T/ipykernel_51640/4019768598.py:56: PerformanceWarning: DataF

  • primer_bases: rows 0–32 done
  • primer_bases: rows 32–64 done
  • primer_bases: rows 64–96 done
  • primer_bases: rows 96–128 done
  • primer_bases: rows 128–160 done
  • primer_bases: rows 160–192 done
  • primer_bases: rows 192–224 done
  • primer_bases: rows 224–256 done
  • primer_bases: rows 256–288 done
  • primer_bases: rows 288–320 done
  • primer_bases: rows 320–352 done
  • primer_bases: rows 352–384 done
  • primer_bases: rows 384–416 done
  • primer_bases: rows 416–448 done
  • primer_bases: rows 448–480 done
  • primer_bases: rows 480–500 done


/var/folders/88/r4q5_g595xs0mq3119k4nxy40000gq/T/ipykernel_51640/4019768598.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_chunk[f"{col}_emb"] = list(all_embs)
/var/folders/88/r4q5_g595xs0mq3119k4nxy40000gq/T/ipykernel_51640/4019768598.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_chunk[emb_cols] = pd.DataFrame(all_embs, columns=emb_cols, index=df_chunk.index)
/var/folders/88/r4q5_g595xs0mq3119k4nxy40000gq/T/ipykernel_51640/4019768598.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually 

 Embedding 'target_region' for 500 rows…


/var/folders/88/r4q5_g595xs0mq3119k4nxy40000gq/T/ipykernel_51640/4019768598.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_chunk[emb_cols] = pd.DataFrame(all_embs, columns=emb_cols, index=df_chunk.index)
/var/folders/88/r4q5_g595xs0mq3119k4nxy40000gq/T/ipykernel_51640/4019768598.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_chunk[emb_cols] = pd.DataFrame(all_embs, columns=emb_cols, index=df_chunk.index)
/var/folders/88/r4q5_g595xs0mq3119k4nxy40000gq/T/ipykernel_51640/4019768598.py:56: PerformanceWarning: DataF

  • target_region: rows 0–32 done
  • target_region: rows 32–64 done
  • target_region: rows 64–96 done
  • target_region: rows 96–128 done
